In [ ]:
import sys
import glob
import torch
import json
import glob
import sys
import os
from sklearn.metrics import roc_curve, auc

import seaborn as sns

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy.stats import norm
from collections import defaultdict
from IPython.display import clear_output

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats

from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utls/')
sys.path.append('../src/model/')

sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability

from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.loading import load_results_with_exclusion_2
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utls.reliability import compute_power_curve
from utls.plotting import plot_power_curve

import DistanceMemoryModel
import encoders

def get_dprime_by_isi(df_per_subject, return_sem=False, return_subjects=False):
    """
    Compute mean d-prime per ISI across subjects, excluding ISI = -1 (lures).

    Args:
        df_per_subject (pd.DataFrame): Output from recompute_dprime_by_isi_per_subject.
        return_sem (bool): Whether to return standard error of the mean.
        return_subjects (bool): Whether to return per-subject d-primes too.

    Returns:
        pd.DataFrame or dict:
            If return_sem=False:
                DataFrame with columns ['isi', 'd_prime']
            If return_sem=True:
                DataFrame with columns ['isi', 'd_prime', 'sem']
            If return_subjects=True:
                Returns a dict with:
                    'summary': summary DataFrame as above,
                    'per_subject': filtered per-subject df
    """
    df_filtered = df_per_subject[df_per_subject["isi"] > -1]

    grouped = df_filtered.groupby("isi")["d_prime"]
    result_df = grouped.mean().reset_index(name="d_prime")

    if return_sem:
        result_df["sem"] = grouped.sem().values

    if return_subjects:
        return {
            "summary": result_df,
            "per_subject": df_filtered.copy()
        }

    return result_df.d_prime.tolist()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = device

base_path = "/om/data/public/audioset/wavs/unbalanced_train_segments_downloads/"

# Load CSV
df = pd.read_csv("/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/stimuli/OVERLAPPED_0.5s_all_4s_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv")

# Filter only the rows that pass the stationarity threshold
df["pass_stationarity"] = df["stationarity_score"] <= -0.6

# Group by full filepath
grouped = df.groupby("filepath")["pass_stationarity"]

# Compute fraction of segments that pass per file
fraction_passing = grouped.mean()

# Keep files where more than 50% of segments pass
files_to_use = fraction_passing[fraction_passing > 0.5].index.tolist()
files_to_use = [base_path+f for f in files_to_use]

print(f"Selected {len(files_to_use)} files where majority of segments are stationary.")


# grabbing example list of sound
sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")
sounds_list = files_to_use
#texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
print(len(ALL_SOUNDS))

tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[2] # "global-music-len120", "atexts-len120" "nhs-region-len120"

base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"

seqs_paths = {"ind-nature-len120": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}

hr_task_name = {"ind-nature-len120": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

In [ ]:
skip = False

if skip: 
    
    exps, seqs, fnames, _ = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                        min_dprime=2,
                                                        min_trials=120,
                                                        skip_len60=True,
                                                        verbose=False,
                                                        return_skipped=skip)
else:
    exps, seqs, fnames = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=False,
                                                    return_skipped=skip)



move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

print("Number of participants used in analysis:", len(exps))


safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only", safe_name)

# where you want the results
CSV_PATH = f"/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/"

ensure_dir(save_dir)
ensure_dir(CSV_PATH)
print(save_dir)

human_results = recompute_dprime_by_isi_per_subject(exps)
human_sensitivity = get_dprime_by_isi(human_results)

clear_output(wait=False)
CSV_PATH = f"/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/isi16_runs.csv"

from sklearn.metrics import mean_squared_error

# Your experiment structure (list of stimulus filepaths for each run)
experiment_list = []
for seq in seqs:
    list_to_add = []
    
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as file:
        data = json.load(file)
   
    for stim in  data['filenames_order']:
        edited_stim_name = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + stim
        list_to_add.append(edited_stim_name)
    
    experiment_list.append(list_to_add)

In [ ]:
pc_dims = 256

NV = 0.2                             # per-dim noise std per drift step (your class convention)
CRIT_MULT = 1.5                          # criterion = CRIT_MULT * NV * sqrt(D)
SEED = 123                               # set to None for stochastic runs
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TIMES_TO_PLOT = [10, 17, 16, 22, 40, 80, 119]   # steps for hist panels

# Map filenames to row indices in X0
# 1) Collect all unique filenames in stable order
seen, all_files = set(), []
for seq in experiment_list:
    for fn in seq:
        if fn not in seen:
            seen.add(fn)
            all_files.append(fn)

name_to_idx = {fn: i for i, fn in enumerate(all_files)}


if which_task == "atexts-len120":

    print("using texture statistiscs")
    pc_texture_model = encoders.AudioTextureEncoderPCA(
        statistics_dict=statistics_set.statistics,
        pc_dims=pc_dims,
        model_params=model_params,
        sr=20000,
        rms_level=0.05,
        duration=2.0,
        device=device
    )
    
    # zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=device)
    # zscore_projector.fit(sounds_list)

else:
    print("using nn embeddings")

    ARCHITECTURES =  ['kell2018', 'resnet50']
    TASKS = ['word', 'speaker', 'audioset', 'word_speaker_audioset']
    LAYERS = {
        'kell2018': [
            'input_after_preproc', 'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc'
        ],
        'resnet50': [
            'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
        ],
    }
    
    networks = []
    for architecture in ARCHITECTURES:
        for task in TASKS:
            for layer in LAYERS[architecture]:
                networks.append({'name': f'{architecture}_{task}',
                                'layer': layer})
    
    
    model_name = ARCHITECTURES[0]
    layer = LAYERS[model_name][-1]
    task  = TASKS[3]
    
    sys.path.append(f'/orcd/data/jhm/001/om2/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')
    print(f'/orcd/data/jhm/001/om2/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')
    
    
    kell2018_layer = encoders.Kell2018Encoder(
        model_name=model_name,
        layer=layer,
        sr=20000,
        rms_level=0.05,
        duration=2.0,
        device='cuda'
    )

    print("running pca")

    pc_kell = encoders.PCASpace(
        encoder = kell2018_layer,
        n_components = 256, 
        device='cuda'
    )

    pc_kell.fit(sounds_list[:400])
    clear_output(wait=False)

    print("zscoring")

    zscore_projector = encoders.ZScoreSpace(pc_kell, device=device)
    zscore_projector.fit(sounds_list[400:])
    clear_output(wait=False)


 
# 2) Use your pipeline to encode clean reps X0: [M,D]
_tmp = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=pc_texture_model,
    noise_variance=float(NV),   # irrelevant for encoding, but ctor requires it
    criterion=0.0,
    device=DEVICE
)
_tmp._fill_memory_bank(all_files)
clear_output(wait=False)

with torch.no_grad():
    X0 = torch.stack([rep.detach().clone().view(-1) for rep in _tmp.memory_bank], dim=0).to(DEVICE)
del _tmp
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Compute per-feature statistics
means = X0.mean(dim=0)
stds = X0.std(dim=0)

# X0 = (X0 - means) / stds

# # Compute per-feature statistics
# means = X0.mean(dim=0)
# stds = X0.std(dim=0)

print("Mean of each feature (should be ~0):")
print(means)
plt.hist(means.detach().cpu().numpy(), label='means from data')

print("Std of each feature (should be ~1):")
print(stds)
plt.hist(stds.detach().cpu().numpy(), label='stds from data')
plt.legend();

In [ ]:
# ---------------------------
# 1) define a common FPR grid
# ---------------------------
fpr_grid = np.linspace(0, 1, 101)  # 101 points between 0 and 1

# ---------------------------
# 2) interpolate each subject's ROC onto that grid
# ---------------------------
all_interp_tprs = []
for exp in exps:
    a = np.array(exp.response, dtype=np.int64)
    b = np.array(exp.repeat == 'true', dtype=np.int64)
    fpr, tpr, _ = roc_curve(b, a, drop_intermediate=True)
    tpr_i = np.interp(fpr_grid, fpr, tpr)  # interpolate
    all_interp_tprs.append(tpr_i)
    plt.plot(fpr, tpr, marker='o', linestyle='-', alpha=0.3)  # each participant

all_interp_tprs = np.vstack(all_interp_tprs)

# ---------------------------
# 3) group mean ± SEM on the grid
# ---------------------------
mean_human_tpr = all_interp_tprs.mean(axis=0)
sem_human_tpr  = all_interp_tprs.std(axis=0) / np.sqrt(all_interp_tprs.shape[0])
human_auc_val  = np.trapz(mean_human_tpr, fpr_grid)

# ---------------------------
# 4) plot group curve with error band
# ---------------------------
plt.plot(fpr_grid, mean_human_tpr, '-k', label=f"Mean Human ROC | AUC={human_auc_val:.3f}")
plt.fill_between(fpr_grid,
                 mean_human_tpr - sem_human_tpr,
                 mean_human_tpr + sem_human_tpr,
                 color='k', alpha=0.3)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

In [ ]:
# ---------------------------
# 1) Define FPR grid for interpolation
# ---------------------------
fpr_grid = np.linspace(0, 1, 101)
all_interp_tprs = []

# ---------------------------
# 2) Loop through participants
# ---------------------------
for exp in exps:
    stimuli   = np.array(exp.stimulus)
    repeats   = np.array(exp.repeat) == 'true'
    isis      = np.array(exp.isi, dtype=float)
    responses = np.array(exp.response, dtype=np.int64)

    eligible_hit_idx = []
    eligible_study_idx = []

    # --- find repeated items whose ISI > 2 ---
    for j, is_rep in enumerate(repeats):
        if is_rep and isis[j] > 2:
            stim = stimuli[j]
            prev_idx = np.where(stimuli[:j] == stim)[0]
            if len(prev_idx) > 0:
                eligible_hit_idx.append(j)
                eligible_study_idx.append(prev_idx[-1])

    # --- collect which stimuli were repeated (ISI>2) ---
    repeated_stims_gt2 = set(stimuli[eligible_hit_idx])

    # --- false alarms: trials that were not repeats and never repeated with ISI>2 ---
    fa_idx = [
        i for i, (stim, is_rep) in enumerate(zip(stimuli, repeats))
        if (not is_rep) and (stim not in repeated_stims_gt2)
    ]

    # skip if no valid data
    if len(eligible_hit_idx) == 0 or len(fa_idx) == 0:
        print(f"[skip] participant {getattr(exp,'subj_id', '?')} (no valid ISI>2 trials)")
        continue

    # --- combine responses & labels ---
    scores = np.concatenate([responses[eligible_hit_idx], responses[fa_idx]])
    labels = np.concatenate([np.ones(len(eligible_hit_idx)), np.zeros(len(fa_idx))])

    # --- compute ROC ---
    fpr, tpr, _ = roc_curve(labels, scores, drop_intermediate=True)
    tpr_i = np.interp(fpr_grid, fpr, tpr)  # interpolate onto grid
    all_interp_tprs.append(tpr_i)

    plt.plot(fpr, tpr, alpha=0.3, marker='o', linestyle='-', label=None)

# ---------------------------
# 3) Group mean ± SEM
# ---------------------------
all_interp_tprs = np.vstack(all_interp_tprs)
mean_tpr = all_interp_tprs.mean(axis=0)
sem_tpr  = all_interp_tprs.std(axis=0) / np.sqrt(all_interp_tprs.shape[0])
auc_val  = np.trapz(mean_tpr, fpr_grid)

# ---------------------------
# 4) Plot group ROC
# ---------------------------
plt.plot(fpr_grid, mean_tpr, '-k', lw=2,
         label=f"Mean Human ROC | ISI>2 | AUC={auc_val:.3f}")
plt.fill_between(fpr_grid, mean_tpr - sem_tpr, mean_tpr + sem_tpr,
                 color='k', alpha=0.3)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Group ROC (ISI > 2)")
plt.legend()
plt.show()

In [ ]:
def run_experiment_at_noise(NV_value):
    """Run memory bank simulation at one noise level and collect distances."""
    hit_dists, fa_dists = [], []

    for seq in experiment_list:
        if len(seq) == 0:
            continue

        # fresh model
        model = DistanceMemoryModel.DistanceMemoryModel(
            encoding_model=zscore_projector,
            noise_variance=float(NV_value),
            criterion=0.0,   # criterion not needed for distance collection
            device=DEVICE
        )
        model.memory_bank = []

        seq_idx = [name_to_idx[fn] for fn in seq]

        bank_set, last_seen = set(), {}

        with torch.no_grad():
            for t, idx_incoming in enumerate(seq_idx, start=1):
                if len(model.memory_bank) > 0:
                    model.apply_noise_to_memory()

                if len(model.memory_bank) > 0:
                    bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                    probe = X0[idx_incoming].view(1, -1)
                    dmin = torch.linalg.norm(bank_mat - probe, dim=1).min().item()

                    if idx_incoming in bank_set:
                        hit_dists.append(dmin)
                    else:
                        fa_dists.append(dmin)

                model.memory_bank.append(X0[idx_incoming].detach().clone())
                bank_set.add(idx_incoming)
                last_seen[idx_incoming] = t

    return np.array(hit_dists), np.array(fa_dists)


def roc_from_dists(hit_dists, fa_dists):
    """Compute ROC from hit and FA distance arrays."""
    if hit_dists.size == 0 or fa_dists.size == 0:
        return None, None, None
    y_true  = np.concatenate([np.ones_like(hit_dists), np.zeros_like(fa_dists)])
    y_score = np.concatenate([hit_dists, fa_dists])
    y_score = -y_score  # lower distance = more match-like
    fpr, tpr, thr = roc_curve(y_true, y_score)
    auc_val = auc(fpr, tpr)
    return fpr, tpr, auc_val


# ---------------- Run for a few noise levels ----------------
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.8]   # choose which NVs you want
roc_curves = {}

for nv in noise_levels:
    hits, fas = run_experiment_at_noise(nv)
    res = roc_from_dists(hits, fas)
    if res[0] is not None:
        fpr, tpr, auc_val = res
        roc_curves[nv] = (fpr, tpr, auc_val)

# ---------------- Plot overlay ----------------
plt.figure(figsize=(6,6))
for nv, (fpr, tpr, auc_val) in roc_curves.items():
    plt.plot(fpr, tpr, label=f"noise={nv} | AUC={auc_val:.2f}")
plt.plot(fpr_grid, mean_human_tpr, label=f"Mean Human ROC | AUC={human_auc_val:.3f}")
plt.fill_between(fpr_grid, mean_human_tpr-sem_human_tpr, mean_human_tpr+sem_human_tpr, alpha=0.3)
plt.plot([0,1], [0,1], 'k--', alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves across noise levels")
plt.legend()
plt.show()

In [ ]:
# ===================== imports & small utils =====================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from collections import defaultdict


# ===================== core helpers =====================
def roc_from_arrays(hit_dists, fa_dists):
    if hit_dists.size == 0 or fa_dists.size == 0:
        return None
    y_true  = np.concatenate([np.ones_like(hit_dists), np.zeros_like(fa_dists)])
    y_score = -np.concatenate([hit_dists, fa_dists])

    fpr, tpr, _ = roc_curve(y_true, y_score, drop_intermediate=False)

    # pad endpoints if needed
    if fpr[0] != 0 or tpr[0] != 0:
        fpr = np.insert(fpr, 0, 0.0)
        tpr = np.insert(tpr, 0, 0.0)
    if fpr[-1] != 1 or tpr[-1] != 1:
        fpr = np.append(fpr, 1.0)
        tpr = np.append(tpr, 1.0)

    return fpr, tpr, auc(fpr, tpr)

def plot_roc(ax, fpr, tpr, label):
    """Plot a single ROC curve on ax."""
    ax.plot(fpr, tpr, label=label)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)

# ===================== experiment runner =====================
def run_experiment_at_noise(
    NV_value: float,
    *,
    X0,                            # [M, D] clean embeddings (torch not required here)
    name_to_idx: dict,
    experiment_list,               # list[list[str]]
    DEVICE="cpu",
    DistanceMemoryModel=None,
    zscore_projector=None
):
    """
    Run sequences once at a given noise variance.
    Returns:
      {
        "hits": np.ndarray,             # all hit dmins (pooled)
        "fas":  np.ndarray,             # all FA dmins (pooled)
        "isi_hit_dists": {isi: [(dmin, t), ...]},
        "fa_by_t": list[list[float]],   # index t-1 -> FA dmins at trial t (pooled across sequences)
        "T_max": int                    # longest sequence length
      }
    """
    # per-run accumulators
    hit_dists, fa_dists = [], []
    isi_hit_dists = defaultdict(list)     # isi -> list of (dmin, t)
    T_max = max((len(seq) for seq in experiment_list), default=0)
    fa_by_t = [[] for _ in range(T_max)]

    for seq in experiment_list:
        if not seq:
            continue

        model = DistanceMemoryModel.DistanceMemoryModel(
            encoding_model=zscore_projector,
            noise_variance=float(NV_value),
            criterion=0.0,
            device=DEVICE
        )
        model.memory_bank = []

        seq_idx = [name_to_idx[fn] for fn in seq]
        bank_set, last_seen = set(), {}

        # iterate trials
        import torch  # local import to avoid leaking torch usage elsewhere
        with torch.no_grad():
            for t, idx_incoming in enumerate(seq_idx, start=1):
                if len(model.memory_bank) > 0:
                    model.apply_noise_to_memory()

                if len(model.memory_bank) > 0:
                    bank_mat = torch.stack([rep.view(-1) for rep in model.memory_bank], dim=0)
                    probe = X0[idx_incoming].view(1, -1)
                    dmin = torch.linalg.norm(bank_mat - probe, dim=1).min().item()

                    if idx_incoming in bank_set:
                        hit_dists.append(dmin)
                        isi = t - last_seen[idx_incoming]  # ISI in trials
                        isi_hit_dists[isi].append((dmin, t))
                    else:
                        fa_dists.append(dmin)
                        fa_by_t[t-1].append(dmin)

                # insert clean probe copy
                model.memory_bank.append(X0[idx_incoming].detach().clone())
                bank_set.add(idx_incoming)
                last_seen[idx_incoming] = t

    return {
        "hits": np.asarray(hit_dists, float),
        "fas":  np.asarray(fa_dists,  float),
        "isi_hit_dists": isi_hit_dists,
        "fa_by_t": fa_by_t,
        "T_max": T_max,
    }

# ===================== per-noise ROC (overall) =====================
def rocs_across_noise(noise_levels, *, X0, name_to_idx, experiment_list, DistanceMemoryModel, zscore_projector, DEVICE="cpu"):
    """
    For each NV, run once and compute overall ROC. Also keep per-ISI & per-trial logs for later slicing.
    Returns:
      curves: {nv: (fpr, tpr, auc)}
      runs:   {nv: run_data_dict_from_runner}
    """
    curves, runs = {}, {}
    for nv in noise_levels:
        run = run_experiment_at_noise(
            nv, X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
            DEVICE=DEVICE, DistanceMemoryModel=DistanceMemoryModel, zscore_projector=zscore_projector
        )
        runs[nv] = run
        res = roc_from_arrays(run["hits"], run["fas"])
        if res is not None:
            curves[nv] = res  # (fpr, tpr, auc)
    return curves, runs

# ===================== ISI-specific ROC & second-half ROC =====================
def roc_for_isi(run_data, isi_value: int):
    """ROC using hits at given ISI vs all pooled FAs."""
    hits_raw = run_data["isi_hit_dists"].get(isi_value, [])
    if not hits_raw or run_data["fas"].size == 0:
        return None
    hits = np.array([d for (d, _t) in hits_raw], float)
    return roc_from_arrays(hits, run_data["fas"])

def roc_for_second_half(run_data):
    """ROC using only trials t > T_max//2. FAs filtered by trial index, hits filtered by their logged t."""
    T_half = run_data["T_max"] // 2
    # hits: collect all (d,t) across ISIs, keep those in second half
    hits = []
    for lst in run_data["isi_hit_dists"].values():
        hits.extend([d for (d, t) in lst if t > T_half])
    hits = np.asarray(hits, float)

    # FAs: use fa_by_t buckets to filter by half
    fas = []
    for t_idx, bucket in enumerate(run_data["fa_by_t"], start=1):
        if t_idx > T_half:
            fas.extend(bucket)
    fas = np.asarray(fas, float)

    return roc_from_arrays(hits, fas)

# ===================== plotting wrappers =====================
def plot_noise_overlays(curves_by_noise, title="ROC curves across noise levels"):
    """Overlay overall ROC per noise."""
    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    for nv, (fpr, tpr, auc_val) in curves_by_noise.items():
        plot_roc(ax, fpr, tpr, label=f"noise={nv:g} | AUC={auc_val:.3f}")
    ax.plot(fpr_grid, mean_human_tpr, label="Mean Human ROC")
    ax.fill_between(fpr_grid, mean_human_tpr-sem_human_tpr, mean_human_tpr+sem_human_tpr, alpha=0.3)
    ax.plot([0,1], [0,1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
    
# ---------- add near your helpers ----------
def roc_full(run_data):
    """ROC using ALL hits and ALL FAs (aggregate across ISIs and time)."""
    return roc_from_arrays(run_data["hits"], run_data["fas"])

# ---------- replace plot_isi_and_half_for_noise with this version ----------
def plot_isi_and_half_for_noise(nv: float, run_data, isis=(1, 17)):
    """
    For a single noise level: overlay AGGREGATE (full), ISI=..., and Second Half ROCs.
    """
    fig, ax = plt.subplots(figsize=(6.2, 6.2))

    # Aggregate (full) ROC
    res_full = roc_full(run_data)
    if res_full is not None:
        fpr, tpr, auc_val = res_full
        plot_roc(ax, fpr, tpr, label=f"Aggregate (full) | AUC={auc_val:.3f}")

    # ISI-specific ROCs
    for k in isis:
        res = roc_for_isi(run_data, k)
        if res is not None:
            fpr, tpr, auc_val = res
            plot_roc(ax, fpr, tpr, label=f"ISI={k} | AUC={auc_val:.3f}")

    # Second-half ROC
    res_half = roc_for_second_half(run_data)
    if res_half is not None:
        fpr, tpr, auc_val = res_half
        plot_roc(ax, fpr, tpr, label=f"Second half | AUC={auc_val:.3f}")

    ax.plot(fpr_grid, mean_human_tpr, label=f"Mean Human ROC | AUC={human_auc_val:.3f}")
    ax.fill_between(fpr_grid, mean_human_tpr-sem_human_tpr, mean_human_tpr+sem_human_tpr, alpha=0.3)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"Noise={nv:g}: Aggregate + ISI + 2nd-half ROCs")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

# ===================== USAGE EXAMPLE =====================
# Assumes you already have: X0 (torch.Tensor [M,D]), name_to_idx (dict), experiment_list (list[list[str]]),
# DistanceMemoryModel (module), zscore_projector (encoder), DEVICE (str).
# Pick the noise levels you want:
noise_levels = [0.1, 0.2, 0.3, 0.4, 0.8, 1]

# 1) Overall ROCs across noise
curves_by_noise, runs_by_noise = rocs_across_noise(
    noise_levels,
    X0=X0, name_to_idx=name_to_idx, experiment_list=experiment_list,
    DistanceMemoryModel=DistanceMemoryModel, zscore_projector=zscore_projector, DEVICE=DEVICE
)
plot_noise_overlays(curves_by_noise, title="ROC across noise levels (overall)")

# 2) For a few selected noise levels, plot ISI=1, ISI=17, and Second Half
for nv_focus in noise_levels:  # choose any subset to inspect
    if nv_focus in runs_by_noise:
        plot_isi_and_half_for_noise(nv_focus, runs_by_noise[nv_focus], isis=(1, 17))

In [ ]:
# ===================== per-noise histogram subplots =====================
import math

def _global_xmax_from_runs(runs_by_noise: dict, noise_levels, q: float = 99.5) -> float:
    """Robust global x-maximum (shared across subplots) from all hits/FAs."""
    pooled = []
    for nv in noise_levels:
        run = runs_by_noise.get(nv)
        if not run:
            continue
        if run["hits"].size:
            pooled.append(run["hits"])
        if run["fas"].size:
            pooled.append(run["fas"])
    if not pooled:
        return 1.0
    arr = np.concatenate(pooled)
    # clip to high percentile to avoid outliers blowing up the axis
    xmax = float(np.percentile(arr, q))
    if not np.isfinite(xmax) or xmax <= 0:
        xmax = float(arr.max(initial=1.0))
    return xmax

def plot_min_distance_subplots(runs_by_noise: dict, noise_levels, bins: int = 40, sharex: bool = True, sharey: bool = True):
    """
    For each noise level, draw a subplot overlaying histograms of min distances:
    - Hits  (in-bank repeats)
    - FAs   (non-repeats)
    All panels share a global x-range for comparability.
    """
    # Determine layout
    n = len(noise_levels)
    if n == 0:
        print("No noise levels to plot.")
        return

    # Shared x-axis limit across all panels (robust)
    xmax = _global_xmax_from_runs(runs_by_noise, noise_levels, q=99.9)

    # Choose a compact grid
    ncols = min(3, n)                          # up to 3 columns looks clean
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(nrows, ncols, figsize=(5.0*ncols, 3.8*nrows), sharex=sharex, sharey=sharey)
    if nrows * ncols == 1:
        axes = np.array([axes])
    axes = axes.ravel()

    # Plot each noise level
    for ax, nv in zip(axes, noise_levels):
        run = runs_by_noise.get(nv)
        if run is None:
            ax.set_visible(False)
            continue

        # Pull arrays; clip to shared xmax to keep scales aligned
        hits = np.clip(np.asarray(run["hits"], float), 0, xmax)
        fas  = np.clip(np.asarray(run["fas"],  float), 0, xmax)

        # Plot normalized histograms; alpha for readable overlap
        if hits.size > 0:
            ax.hist(hits, bins=bins, range=(0, xmax), density=True, alpha=0.5, label="Hits")
        if fas.size > 0:
            ax.hist(fas,  bins=bins, range=(0, xmax), density=True, alpha=0.5, label="FAs")

        ax.set_title(f"noise={nv:g}")
        ax.grid(True, alpha=0.2)

        # Only put legend on first populated subplot to reduce clutter
        # (If you prefer each panel to have its own legend, remove this block and always call ax.legend().)
        if nv == noise_levels[0]:
            ax.legend(loc="upper right", fontsize=8)

    # Hide any extra axes if grid has more slots than noise levels
    for ax in axes[len(noise_levels):]:
        ax.set_visible(False)

    # Global labels
    fig.supxlabel("min distance", fontsize=11)
    fig.supylabel("density", fontsize=11)
    fig.suptitle("Min-distance histograms by noise level (Hits vs FAs)", y=0.995, fontsize=13)
    plt.tight_layout()
    plt.show()

# ---------- call it ----------
plot_min_distance_subplots(runs_by_noise, noise_levels, bins=50)

In [ ]:
## need to look at distributions for just ISI=16
# get analytic forms for that distribution... then you can compute the analytic ROC curves

In [ ]:
runs_by_noise